In [ ]:
# Resolve project inputs consistently when Jupyter opens this subdirectory.
import os

from paths import ROOT

os.chdir(ROOT)
import functools
import json
import pickle

# Import the original monkey preprocessing dependencies and configure output paths.
from collections import OrderedDict, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.io as sio

from analysis.monkey import get_collective_hs

data_dir = Path("artifacts/dpx/monkey/")
data_dir.mkdir(parents=True, exist_ok=True)
Path("artifacts/dpx/monkey").mkdir(parents=True, exist_ok=True)

In [ ]:
# Parse rewarded trials into epoch features and aligned neural time courses.
dt = 50
# dt = 25

color_pal_cueprobes = {
    "AX": "#F7AD57",
    "AY": "#F1565C",
    "BX": "#AA5DA5",
    "BY": "#5598CA",
}

cue_ids = {"A": 0, "B": 1}
probe_ids = {"X": 0, "Y": 1}
cue_probes = ["AX", "AY", "BX", "BY"]

dt2phase_ids = {
    4: {
        "fix": (0, 125),
        "cue": (125, 375),
        "delay": (375, 625),
        "probe": (625, 750),
    },
    25: {
        "fix": (0, 20),
        "cue": (20, 60),
        "delay": (60, 100),
        "probe": (100, 120),
    },
    50: {
        "fix": (0, 10),
        "cue": (10, 30),
        "delay": (30, 50),
        "probe": (50, 60),
    },
    100: {
        "fix": (0, 5),
        "cue": (5, 15),
        "delay": (15, 25),
        "probe": (25, 30),
    },
    250: {
        "fix": (0, 2),
        "cue": (2, 6),
        "delay": (6, 10),
        "probe": (10, 12),
    },
}

trial_info_dict = {
    1: "AX",
    2: "AY",
    3: "BX",
    4: "BY",
}


sensory_mapping = {
    0: "no",
    1: "A",
    2: "B",
    3: "X",
    4: "Y",
}

dt2phases = dt2phase_ids[int(dt)]


def filter_data(df_all, hs_dict_all, nnids, nnids_shared):
    filtered_cols = (
        ["drug_cond", "cue_probe", "cue", "probe", "target", "rt"]
        + [f"b_{idx}" for idx in nnids_shared]
        + [f"c_{idx}" for idx in nnids_shared]
        + [f"d_{idx}" for idx in nnids_shared]
        + [f"p_{idx}" for idx in nnids_shared]
        + [f"r_{idx}" for idx in nnids_shared]
    )
    df_all = df_all[filtered_cols].copy()
    nnids = nnids.tolist()
    nnid_indices = [nnids.index(nid) for nid in nnids_shared]

    for k, v in hs_dict_all.items():
        hs_dict_all[k] = {
            "raw": v["raw"][:, :, nnid_indices],
            "mean": v["mean"][:, nnid_indices],
        }
    return df_all, hs_dict_all


def parse_trial_data(episode_dat, neuron_ids: list, all_neuron_ids: list):
    """Parse the episode data cell.

    Args:
        episode_dat (np.ndarray): A time frame matrix whose
            rows are differnet time slices and columns are different variables.
        neuron_ids: the ids of neurons in this trial.
        all_neuron_ids: the ids of neurons we want to keep.
    Returns:
        tuple: (cue, probe, hs, info_type, row).
            cue (str): cue type.
            probe (str): probe type.
            hs (np.ndarray): neural activity matrix.
            info_type (str): trial type.
            row: a list of features including cue, probe, target, rt, and
                mean activity of all neurons in each phase.
    """
    info_type = episode_dat[:, 0]
    assert len(np.unique(info_type)) == 1
    info_type = trial_info_dict[info_type[0]]
    if info_type == "AX":
        target = 1
    else:
        target = 0
    cue, probe = list(info_type)
    cue_id = cue_ids[cue]
    probe_id = probe_ids[probe]

    timestamp = episode_dat[:, 1]

    start_idx = np.argwhere(timestamp >= -500)[:, 0][0]

    timestamp = timestamp[start_idx:]
    _sensory_input = episode_dat[start_idx:, 2]
    action = episode_dat[start_idx:, 3]

    hs = episode_dat[start_idx:, 4:]
    nid_indices = [neuron_ids.index(nid) for nid in all_neuron_ids]

    hs = hs[:, nid_indices]

    response_idx = np.argwhere(action != 0)[:, 0][0]
    rt = response_idx - dt2phases["probe"][0]

    beforecue_h_mean = hs[dt2phases["fix"][0] : dt2phases["fix"][1]].mean(axis=0).tolist()

    cue_h_mean = hs[dt2phases["cue"][0] : dt2phases["cue"][1]].mean(axis=0).tolist()

    delay_h_mean = hs[dt2phases["delay"][0] : dt2phases["delay"][1]].mean(axis=0).tolist()

    probe_h_mean = hs[dt2phases["probe"][0] : dt2phases["probe"][1]].mean(axis=0).tolist()

    ressponse_h_mean = hs[dt2phases["probe"][0] : -1].mean(axis=0).tolist()

    row = (
        [info_type, cue_id, probe_id, target, rt]
        + beforecue_h_mean
        + cue_h_mean
        + delay_h_mean
        + probe_h_mean
        + ressponse_h_mean
    )
    return (cue, probe, hs, info_type, row)


def parse_dataf(d_f):
    _, nt = d_f["ens_trials"].shape
    assert _ == 1, "parse data failed as the shape of ens_trials " + f"is {d_f['ens_trials'].shape}"

    df_all = []
    trial_ids = []
    drug_conds = []
    hs_dict = defaultdict(list)
    neuron_idss = []
    for trial_idx in range(nt):
        trial_data = d_f["ens_trials"][0, trial_idx]
        neuron_ids = trial_data[0, 0][3].reshape(-1).tolist()
        neuron_idss.append(neuron_ids)

    all_neuron_ids = sorted(list(functools.reduce(lambda x, y: set(x) & set(y), neuron_idss)))

    for trial_idx in range(nt):
        trial_data = d_f["ens_trials"][0, trial_idx]
        episode_data = trial_data[0, 0][0]
        # print(episode_data.shape)

        rewarded = trial_data[0, 0][1][0, 0]
        _ens_num = trial_data[0, 0][2][0, 0]
        neuron_ids = trial_data[0, 0][3].reshape(-1).tolist()
        if rewarded == 0:
            continue

        trial_id = trial_data[0, 0][4].reshape(-1).item()
        drug_cond = trial_data[0, 0][5].reshape(-1).item()
        trial_ids.append(trial_id)
        drug_conds.append(drug_cond)

        (cue, probe, hs, info_type, row) = parse_trial_data(
            episode_data, neuron_ids, all_neuron_ids
        )

        hs_dict[(cue, probe)].append(hs)
        df_all.append([drug_cond, trial_id] + row)

    nids = np.array(all_neuron_ids)
    cols = (
        ["drug_cond", "trial_id", "cue_probe", "cue", "probe", "target", "rt"]
        + [f"b_{idx}" for idx in nids]
        + [f"c_{idx}" for idx in nids]
        + [f"d_{idx}" for idx in nids]
        + [f"p_{idx}" for idx in nids]
        + [f"r_{idx}" for idx in nids]
    )
    df_all = pd.DataFrame(df_all, columns=cols)
    df_all.sort_values(by=["trial_id"], ascending=True, inplace=True)
    df_all.drop(columns=["trial_id"], inplace=True)

    return df_all, hs_dict, nids


def preprocess_dataf(datf):
    df_all, hs_dict, nnids = parse_dataf(sio.loadmat(datf))

    hs_dict_all = dict()
    for cue in cue_ids.keys():
        for probe in probe_ids.keys():
            lengths = [hs.shape[0] for hs in hs_dict[(cue, probe)]]
            nids = [hs.shape[1] for hs in hs_dict[(cue, probe)]]
            err_info = f"In {datf}, Number of neurons \
                is not the same, they are {nids}"
            assert len(np.unique(nids)) == 1, err_info
            nids = nids[0]
            max_length = max(lengths)

            hsall = np.zeros((len(hs_dict[(cue, probe)]), max_length, nids))
            for hs_idx, hs in enumerate(hs_dict[(cue, probe)]):
                hsall[hs_idx, 0 : hs.shape[0], :] = hs

            hs_dict_all[(cue, probe)] = {
                "raw": hsall.copy(),
                "mean": hsall.mean(axis=0),
            }
    return df_all, hs_dict_all, nnids

In [ ]:
# Pair balanced and prepotent sessions and retain their shared recorded neurons.
dat_exp = "artifacts/dpx/monkey/stats.mat"

dataf = sio.loadmat(dat_exp)["statsOut"][:, [3, 17]].astype(int)
exp_ntypes = np.unique(dataf[:, 1])
nids_types = {ntype: dataf[dataf[:, 1] == ntype, 0] for ntype in exp_ntypes}


neuron_groups = OrderedDict()
# Preserve historical session identity and traversal order after filename migration.
sessions = json.loads(Path("artifacts/dpx/monkey/sessions.json").read_text())
for session in sessions:
    p = Path(session["path"])
    trial_type = session["condition"]
    neuron_group = session["group"]
    if neuron_group not in neuron_groups:
        neuron_groups[neuron_group] = dict()
    if trial_type in neuron_groups[neuron_group].keys():
        raise ValueError(f"Duplicate session: {neuron_group}/{trial_type}")
    neuron_groups[neuron_group][trial_type] = p
# neuron_groups is mapping from neuron group to the file paths of pre and bal trials


exp_data_groups = OrderedDict()
nnids_shareds = []
nnids_shared_counts = []
df_all_pre_all = []
for ng_idx, (ng, ng_f) in enumerate(neuron_groups.items()):
    if ng not in exp_data_groups:
        exp_data_groups[ng] = dict()

    assert "pre" in ng_f and "bal" in ng_f

    # A failed session invalidates this preprocessing run; propagate its original error.
    df_all_pre, hs_dict_all_pre, nnids_pre = preprocess_dataf(ng_f["pre"])
    df_all_bal, hs_dict_all_bal, nnids_bal = preprocess_dataf(ng_f["bal"])

    nnids_shared = np.intersect1d(
        nnids_pre, nnids_bal
    )  # select neurons that are in both pre and bal trials

    # assert (nnids_shared==nnids_pre).all() is np.True_
    # filter the data so that only shared neuron recordings are kept
    df_all_pre, hs_dict_all_pre = filter_data(df_all_pre, hs_dict_all_pre, nnids_pre, nnids_shared)
    df_all_bal, hs_dict_all_bal = filter_data(df_all_bal, hs_dict_all_bal, nnids_bal, nnids_shared)
    df_all_pre["ng_idx"] = ng_idx
    df_all_pre_all.append(df_all_pre.copy())

    exp_data_groups[ng]["pre"] = (df_all_pre, hs_dict_all_pre, nnids_shared)
    exp_data_groups[ng]["bal"] = (df_all_bal, hs_dict_all_bal, nnids_shared)
    exp_data_groups[ng]["nnids"] = nnids_shared

    nnids_shareds.append(nnids_shared.tolist())
    nnids_shared_counts.append(len(nnids_shared))
    pass


data_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
# Preserve the original final-group table export used by the matching figure.
df_all_pre.to_pickle("artifacts/dpx/monkey/table_pre.pkl")
df_all_bal.to_pickle("artifacts/dpx/monkey/table_bal.pkl")

In [ ]:
# Map the precomputed neuron classes onto the concatenated population indices.
nnid_idx_types_used = defaultdict(list)
n_id_offset = 0
for ng_idx, ng in enumerate(exp_data_groups.keys()):
    nnids = exp_data_groups[ng]["nnids"]

    for ntype in nids_types.keys():
        nids_common = np.intersect1d(nnids, nids_types[ntype])
        if len(nids_common) > 0:
            nnid_idx_types_used[ntype] += (
                np.where(np.in1d(nnids, nids_common))[0] + n_id_offset
            ).tolist()
        pass

    n_id_offset += len(nnids)
nnid_idx_types_used = dict(nnid_idx_types_used)
for ntype in nnid_idx_types_used.keys():
    nnid_idx_types_used[ntype] = np.array(nnid_idx_types_used[ntype])

In [ ]:
# Concatenate session activity for the original switch-neuron population figure.
(
    collective_exp_hs_dict_mean_list_pre,
    collective_exp_hs_dict_mean_list_bal,
    collective_exp_hs_dict_raw_lists_pre,
    collective_exp_hs_dict_raw_lists_bal,
    nnids_all,
) = get_collective_hs(exp_data_groups)

In [ ]:
# Save the selected monkey population with the original 50-ms time resolution.

pickle.dump(
    {
        "pre": {
            "AX": collective_exp_hs_dict_mean_list_pre[("A", "X")]["mean"],
            "AY": collective_exp_hs_dict_mean_list_pre[("A", "Y")]["mean"],
            "BX": collective_exp_hs_dict_mean_list_pre[("B", "X")]["mean"],
            "BY": collective_exp_hs_dict_mean_list_pre[("B", "Y")]["mean"],
        },
        "bal": {
            "AX": collective_exp_hs_dict_mean_list_bal[("A", "X")]["mean"],
            "AY": collective_exp_hs_dict_mean_list_bal[("A", "Y")]["mean"],
            "BX": collective_exp_hs_dict_mean_list_bal[("B", "X")]["mean"],
            "BY": collective_exp_hs_dict_mean_list_bal[("B", "Y")]["mean"],
        },
        "ids2plot": nnid_idx_types_used[1],
        # "dt": dt*2,# as we used block reduce to smooth the data
        "dt": dt,  # as we used block reduce to smooth the data
    },
    open(data_dir / "population.pkl", "wb"),
)